# Recursive MHM in identical leaf spaces

This notebook consumes the ten current Cartesian Q2/P1 acquisitions: five outer meshes and both zero and affine weak Dirichlet pressure. Each outer macrocell has four inner macrocells, each with four fine Q2 quadrilaterals. Recursive and flat MHM share these leaf and trace spaces. The reference independently assembles the complete original system with FEniCS Basix 0.9.0; its attributed SciPy solver arithmetic is shared.

The measured smooth-case rates and finite rank/gauge controls do not establish a mesh-uniform inf-sup bound or transfer a single-cell Pk theorem to Q2 on a fine submesh. Numerical bases, orientation maps and evaluation tables are part of each archive. This notebook replays the selected saved field without solving or tabulating a new basis.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/9fb1174400c6deea7df6424cd459edc6a39c4b30d701c73e75db40315326e962/36_recursive_mhm-companion.zip"
COMPANION_SHA256 = "9fb1174400c6deea7df6424cd459edc6a39c4b30d701c73e75db40315326e962"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/36_recursive_mhm.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


## Write the hierarchy with the variational API

The live example below has one outer Cartesian macrocell, four child macrocells and continuous Q2 leaves. Each outer edge has two P1 segments. Child boundary restrictions include the normal orientation explicitly. This is the same declared spatial construction as the archived study; the archived convergence acquisitions retain their own provenance.

A leaf declares

$$
(K\nabla p,\nabla v)+\langle\lambda,v\rangle=(f,v).
$$

If a child's assembled equation is $A x=f$, its parent local equation uses the exact trace constraint

$$
\begin{bmatrix}-A&E\\E^T&0\end{bmatrix}
\begin{bmatrix}x\\\mu\end{bmatrix}
+\begin{bmatrix}0\\-M\end{bmatrix}\lambda
=\begin{bmatrix}-f\\0\end{bmatrix}.
$$

`NestedEquations` declares that equality constraint and the child's physical constant kernel. `leaf_moment` represents a finest-scale integral in the executed global coordinates, including source offsets, for a physical Neumann gauge.


In [ ]:
import os
HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
import inspect
import numpy as np
from pymhm import assemble, leaf_moment
from examples.nested_formulation import leaf_equations, parent_equations, nested_problem
from pymhm.fem.scalar.quadrilateral import qk_space

# These functions define the actual blocks consumed below.
print(inspect.getsource(leaf_equations))
print(inspect.getsource(parent_equations))


In [ ]:
def exact_patch(points):
    return 1 + points[:, 0] + np.sum(points**2, axis=1)

# -Delta(p)=-4 follows independently from the quadratic exact field.
problem, discretization = nested_problem(source=-4.0, dirichlet=exact_patch)
system = assemble(problem)
solution = system.solve()
child = solution.children[0]
child_mesh = discretization.mesh.submesh(0, 2)
maximum_leaf_error = max(
    np.max(abs(field - exact_patch(qk_space(child_mesh.submesh(cell, 2), 2)[1])))
    for cell, field in enumerate(child.fields)
)
row, offset = leaf_moment(system, lambda metadata: metadata[1])
physical_integral = row @ np.r_[solution.trace, *solution.coarse] + offset
assert maximum_leaf_error < 2e-11
assert abs(physical_integral - (1.5 + 2 / 3)) < 2e-12
assert solution.raw_residual < 1e-10
{
    "finest_pressure_error": float(maximum_leaf_error),
    "physical_pressure_integral": float(physical_integral),
    "reconstructed_child_fields": len(child.fields),
    "original_equation_residual": solution.raw_residual,
}


## Reproduce the refinement study's physical problem

The following acquisition uses the sine source and the same Q2 leaf spaces as
the refinement study, with homogeneous and nonhomogeneous weak Dirichlet data.
Its importable acquisition helper assembles the explicit public providers,
compares recursive and flat fields, and reports integrated pressure and raw
flux differences. The quadratic calculation above is a separate exact patch.


In [ ]:
from examples.verify_nested import acquire

live_refinement = [acquire(1, nonhomogeneous=value)[0] for value in (False, True)]
for row in live_refinement:
    print(row)

# Named fields descend through the recovered hierarchy in macrocell order.
pressure_fields = solution.field("pressure", recursive=True)
print("Recovered leaf pressure fields:", len(pressure_fields))


## Retained recursive refinement measurements

The source-only record below identifies the executed Q2/P1 refinement and
nonhomogeneous boundary study. The subsequent original coefficient replay is
optional and requires `python -m scripts.run_notebooks /path/to/36_recursive_mhm.ipynb --historical`, the selected field archive
and the attributed external native-verification acquisition. The public
`examples.verify_nested` module acquires a fresh complete analytical refinement
with its own execution provenance; it does not recreate those original hashes.

In [ ]:
import json
retained = json.loads((ROOT / "examples/results/nested.json").read_text())
print("Retained refinement fields:", sorted(retained))
print("Current public full acquisition: python -m examples.verify_nested")


In [ ]:
if HISTORICAL_REPLAY:
    from pathlib import Path
    import hashlib
    import json
    import sys
    
    import numpy as np
    from IPython.display import Image, display
    
    from examples.nested_field_archive import display_fields, read_archive
    
    record_path = root / "examples/results/nested.json"
    record = json.loads(record_path.read_text())
    assert record["schema"] == "pymhm-recursive-study-v2"
    assert len(record["rows"]) == 10
    assert {(row["n"], row["boundary_case"]) for row in record["rows"]} == {
        (n, boundary) for n in (1, 2, 4, 8, 16)
        for boundary in ("sine_zero_dirichlet", "affine_plus_sine")
    }
    table = [
        {key: row[key] for key in (
            "n", "boundary_case", "outer_global_dofs", "flat_global_dofs", "pressure_l2", "flux_l2"
        )}
        for row in record["rows"]
    ]
    table
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/36_recursive_mhm.ipynb --historical with the attributed archives.")


In [ ]:
if HISTORICAL_REPLAY:
    assert record["maximum_original_source_and_constraint_relative_residual"] < 1e-10
    assert record["maximum_recursive_flat_pressure_and_raw_flux_norm_difference"] < 1e-9
    assert record["maximum_norm_q8_q10_relative_sensitivity"] < 1e-9
    assert max(row["relative_leaf_difference"] for row in record["rows"]) < 1e-11
    
    native_path = record_path.parent / record["native_verification"]["record"]
    assert hashlib.sha256(native_path.read_bytes()).hexdigest() == record["native_verification"]["record_sha256"]
    native = json.loads(native_path.read_text())
    assert native["complete"] and native["cases"] == 10
    assert native["acquisition_source_identity_sha256"] == record["source_identity_sha256"]
    assert native["maximum_native_original_relative"] < native["criteria"]["original"]
    assert native["maximum_candidate_in_native_original_relative"] < native["criteria"]["original"]
    assert native["maximum_same_space_physical_field_relative"] < native["criteria"]["physical_fields"]
    assert native["maximum_represented_operator_relative_difference"] < native["criteria"]["represented_operator_identity"]
    assert native["maximum_native_norm_q8_q10_relative_sensitivity"] < native["criteria"]["norm_integration"]
    assert native["bitwise_data_only_replay_and_comparison_threads_1_2"]
    {
        "observed_final_rates": {boundary: rates[-1] for boundary, rates in record["observed_rates"].items()},
        "native_original": native["maximum_native_original_relative"],
        "candidate_in_native_original": native["maximum_candidate_in_native_original_relative"],
        "physical_field_difference": native["maximum_same_space_physical_field_relative"],
        "reference": native["reference"],
    }
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/36_recursive_mhm.ipynb --historical with the attributed archives.")


In [ ]:
if HISTORICAL_REPLAY:
    selected = record["display_field"]
    archive = record_path.parent / selected["archive"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == selected["archive_sha256"]
    acquisition, arrays = read_archive(archive)
    assert acquisition["n"] == selected["n"] == 4
    assert acquisition["boundary_case"] == selected["boundary_case"] == "sine_zero_dirichlet"
    points, pressure, raw_flux = display_fields(arrays)
    assert points.shape == (64, 289, 2)
    assert pressure.shape == (64, 289)
    assert raw_flux.shape == (64, 289, 2)
    assert all(np.all(np.isfinite(value)) for value in (points, pressure, raw_flux))
    {
        "acquisition_uuid": acquisition["acquisition_uuid"],
        "saved_one_sided_samples": pressure.size,
        "coefficient_precision_bits": acquisition["coefficient_precision_bits"],
        "actual_display_table_digest": acquisition["arrays_sha256"]["display_basis"],
        "pressure_range": (float(pressure.min()), float(pressure.max())),
    }
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/36_recursive_mhm.ipynb --historical with the attributed archives.")


The left convergence panel measures absolute analytical L2 errors. The right panel compares recursive and flat physical fields, independently of coefficient differences. The final pressure and raw-flux orders approach three and two in both boundary problems.

The spatial panels use the selected homogeneous n=4 archive. Dark lines mark actual outer macrofaces and thin lines mark actual inner macrofaces on analytical, numerical and error panels. Raw Darcy flux is minus the one-sided pressure gradient; it is distinct from an H(div) field and from the oriented normal-flux multiplier. Nine independent colorbars retain each panel's scale, including its pointwise error. Full broken H1 norms in the native record include both pressure L2 and gradient L2 terms.


In [ ]:
if HISTORICAL_REPLAY:
    figure_folder = root / "docs/figures/nested"
    provenance = json.loads((figure_folder / "provenance.json").read_text())
    assert provenance["acquisition_uuid"] == acquisition["acquisition_uuid"]
    assert provenance["input_archive_sha256"] == selected["archive_sha256"]
    assert provenance["input_record_sha256"] == hashlib.sha256(record_path.read_bytes()).hexdigest()
    for name in ("convergence", "fields"):
        figure = figure_folder / f"{name}.png"
        assert hashlib.sha256(figure.read_bytes()).hexdigest() == provenance["figures_sha256"][figure.name]
        display(Image(filename=str(figure), width=960))
else:
    print("Original external field replay is optional; use python -m scripts.run_notebooks darcy/36_recursive_mhm.ipynb --historical with the attributed archives.")


The current analytical geometry is a declared equivalence study, distinct from a literal historical mesh or figure reproduction. Normal macroface equilibrium does not establish fine-cell conservation of the raw gradient. Reduced outer unknown counts measure condensation; setup, archival and verification costs do not demonstrate a runtime speedup.

See `docs/cases/nested.md` for the physical equations, finite rank/gauge argument, complete original-system criteria, source attribution and commands for a fresh acquisition and data-only rendering. External reference sources remain outside the versioned repository and release artifacts.
